In [1]:
#Question 1: Explain boosting in your own words. How is it different from bagging?

- Boosting

Boosting creates a sequence of weak learners (typically shallow decision trees) iteratively, such that each new learner is trained to fix the mistakes made by the previous ones.

The basic outline:

Train a simple model on the dataset.
Identify where the model made a mistake (either misclassified points or the residuals in regression case).
Train the next model on top of those points or residuals — either by giving more weight to them (AdaBoost) or by simply fitting the new model to the residuals (Gradient Boosting).
Repeat the process many times.
Combine the results of all the models into the final model as a weighted average, such that better performing models get higher weight.

As each model learns to fix the mistakes made by the previous ensemble, the models themselves are highly dependent on each other.

Bagging (Bootstrap Aggregating)

Bagging, too, constructs an ensemble of models; the difference lies in its approach:

Take multiple bootstrap samples from the training data.
Train an independent and separate model in parallel on each sample; typically, it would be a fairly complex/deep model, e.g., an unpruned decision tree.
Average all the models' predictions for classification or regression tasks.

The independent nature of each

In [2]:
#Question 2: What is dimensionality reduction? Why is it important?


- Dimensionality reduction is the process of taking data with a large number of features (dimensions) and transforming or projecting it into a lower-dimensional space, while trying to preserve as much of the meaningful structure or information in the data as possible.

Instead of throwing away features arbitrarily, most methods either:

Select a subset of the most useful original features (feature selection), or
Transform the original features into a new, smaller set of features that are combinations of the originals (feature extraction).

Common techniques:

PCA (Principal Component Analysis) — finds new axes (principal components) that capture the most variance in the data, and projects data onto the top few.
t-SNE / UMAP — nonlinear techniques mainly used for visualizing high-dimensional data in 2D/3D by preserving local neighborhood structure.
LDA (Linear Discriminant Analysis) — finds a projection that best separates known classes.
Autoencoders — neural networks trained to compress data into a smaller latent representation and reconstruct it back.

In [3]:
# Question 3:What is variance in PCA and why do we maximize it?


- Variance in PCA

In PCA, "variance" refers to how spread out the data points are along a particular direction (axis) in the feature space.

When PCA looks for principal components, it's searching for directions in the data where, if you projected all your points onto that single line, the projected points would be as spread out as possible — i.e., that direction captures the most variance.

Concretely:

The first principal component is the direction along which the data varies the most.
The second principal component is the direction, orthogonal (perpendicular) to the first, that captures the next-most variance.
This continues for as many components as there are dimensions, each one orthogonal to all the previous ones.

Mathematically, PCA finds these directions by computing the eigenvectors of the data's covariance matrix; the corresponding eigenvalues tell you how much variance is captured along each eigenvector.

In [4]:
# Question 4: Why does UMAP perform well for visualization?

- Question 4: Why does UMAP perform well for visualization?

Why UMAP performs well for visualization
UMAP (Uniform Manifold Approximation and Projection) is popular for visualization because it does a good job of preserving both local and, to a reasonable extent, global structure when squashing high-dimensional data down into 2D or 3D — and it does this efficiently.

Here's the breakdown of what makes it work well:

1. It's built on manifold learning assumptions
UMAP assumes the high-dimensional data actually lies on (or near) a lower-dimensional manifold embedded in that high-dimensional space — like a crumpled sheet of paper in 3D space that's "really" 2D. Instead of preserving raw distances (like PCA does), UMAP tries to preserve the underlying manifold's topology — the neighborhood relationships between points.

2. It constructs a graph of local relationships
UMAP starts by building a weighted graph where each point is connected to its nearest neighbors, with edge weights reflecting how "close" points are in a fuzzy, probabilistic sense. This captures the local neighborhood structure very well — points that are truly close together in high dimensions tend to stay close together in the 2D embedding.

3. It optimizes a low-dimensional layout to match that graph
It then constructs a similar graph in low-dimensional space and uses gradient descent (minimizing cross-entropy between the two graphs' fuzzy topological structures) to arrange points so the low-dimensional graph resembles the high-dimensional one as closely as possible. This is what produces those tight, visually distinct clusters UMAP is known for.

4. It balances local and global structure better than t-SNE
t-SNE is excellent at preserving local neighborhoods (points that are close stay close) but tends to distort global structure — the relative distances between clusters in a t-SNE plot are often not meaningful.
UMAP, due to its use of a more theoretically grounded loss function and initialization (often via spectral embedding), tends to better preserve the relative positioning of clusters relative to each other, so the "big picture" layout is somewhat more trustworthy.
5. It's fast and scalable
UMAP is significantly faster than t-SNE on large datasets due to more efficient nearest-neighbor search (via approximate methods like NN-descent) and optimization. This means you can visualize much larger datasets in less time, which matters a lot in practice.

6. It produces visually crisp, well-separated clusters
Because of how it optimizes the embedding (attracting neighbors, repelling non-neighbors), UMAP tends to produce plots with clearly separated, tight clusters — which is visually satisfying and often reveals structure (like distinct classes or subpopulations) clearly.

In [5]:
#Question 5: Why is KNN called a lazy learner?


- KNN (K-Nearest Neighbors) is called a lazy learner because it does essentially no work during the training phase — it just memorizes the training data instead of learning a generalized model from it.

Lazy vs. Eager learning

Eager learners (like logistic regression, decision trees, neural networks, SVMs) do the heavy lifting upfront:

During training, they analyze the entire dataset and build an explicit, generalized model — weights, splits, boundaries, etc.
Once trained, that model is compact and prediction is fast, because all the "learning" has already been baked into a fixed set of parameters.
Training is slow/expensive; prediction is fast.

Lazy learners (like KNN) postpone all the real work until prediction time:

During "training," KNN does nothing more than store the training dataset in memory. There's no optimization, no parameter fitting, no abstraction built from the data.
All the actual computation happens when you ask it to predict on a new point: it has to calculate the distance from that point to every (or many) training points, find the K closest ones, and then vote/average among them.
Training is essentially instant; prediction is slow, especially as the dataset grows.
Why this distinction matters
No generalization step — KNN never builds an abstract representation of the underlying pattern in the data (no equation, no tree, no boundary function). It just keeps the raw examples around and relies on local comparisons at prediction time.
Computation is deferred — the "learning" happens reactively, on-demand, for each individual query, rather than proactively during a dedicated training phase.
Cost shifts to inference time — because there's no compression of the data into a model, every prediction requires scanning (or efficiently searching, e.g., via KD-trees/ball trees) through the stored training data, making prediction computationally expensive relative to eager learners, especially with large datasets or high dimensions.
Memory-heavy — since it needs to retain the entire training set, memory usage scales directly with the size of the training data, unlike eager learners that discard the raw data after training and just keep the learned parameters.

In [6]:
#Question 6:Why is feature scaling important in KNN?

- KNN makes predictions based purely on distance (usually Euclidean distance) between points. That means the scale of each feature directly determines how much influence it has on the outcome — and this is exactly where things can go wrong if features aren't scaled.

The core problem

Imagine a dataset with two features:

Age: ranges from 0–100
Income: ranges from 0–200,000

If  compute Euclidean distance directly on raw values:

𝑑
=
(
𝑎
𝑔
𝑒
1
−
𝑎
𝑔
𝑒
2
)
2
+
(
𝑖
𝑛
𝑐
𝑜
𝑚
𝑒
1
−
𝑖
𝑛
𝑐
𝑜
𝑚
𝑒
2
)
2
d=
(age
1
	​

−age
2
	​

)
2
+(income
1
	​

−income
2
	​

)
2
	​


The income term will completely dominate the distance calculation, simply because its numbers are on a much larger scale — not because it's actually more important. A difference of 20 years in age contributes almost nothing to the total distance compared to a difference of $20,000 in income, even if age is the more meaningful predictor for the task.

As a result, KNN would effectively become "K-Nearest-Income-Neighbors," largely ignoring age — which has nothing to do with the real relevance of that feature and everything to do with arbitrary units of measurement.

Why this is especially critical for KNN

Unlike models such as decision trees (which split on thresholds per feature independently and are scale-invariant), KNN combines all features together into a single distance metric. So:

Every feature contributes to distance simultaneously — there's no way for KNN to "learn" that a feature's raw scale shouldn't matter; it just adds up whatever numbers you give it.
No inherent weighting mechanism — KNN doesn't learn feature importance the way linear models learn coefficients. Scaling is often the only lever you have to make sure features contribute fairly.
Distance-based methods in general are sensitive to this — the same issue applies to K-Means clustering, SVMs with RBF kernels, PCA, and other distance/variance-based methods.
Common scaling techniques
Standardization (Z-score scaling): transforms features to have mean 0 and standard deviation 1:
𝑥
′
=
𝑥
−
𝜇
𝜎
x
′
=
σ
x−μ
	​

Min-Max normalization: rescales features to a fixed range, typically [0, 1]:
𝑥
′
=
𝑥
−
𝑥
𝑚
𝑖
𝑛
𝑥
𝑚
𝑎
𝑥
−
𝑥
𝑚
𝑖
𝑛
x
′
=
x
max
	​

−x
min
	​

x−x
min
	​

	​


Either approach puts features on a comparable footing so no single feature dominates the distance calculation just because of the units it happens to be measured in.

Intuition in one line: since KNN literally measures "closeness" using raw feature values, unscaled features with larger numeric ranges silently hijack the distance calculation — feature scaling makes sure similarity is judged based on the actual pattern in the data, not the arbitrary units each feature happens to use.

In [7]:
#Question 7: Dataset: Use scikit-learn breast cancer dataset


- Read 3 files
Even without using the labels during PCA (it's unsupervised), the malignant and benign classes separate quite cleanly along PC1 — a good sign that the dominant axis of variance is closely tied to tumor malignancy.

View PC1 feature loadings plot pc1_loadings.png
Results
Dataset: 569 samples × 30 features, standardized with StandardScaler before PCA (mandatory here since features like "mean area" and "mean smoothness" are on wildly different scales).

How many components explain 95% variance?
10 components are needed to reach ~95.16% cumulative explained variance (see the cumulative variance plot — the green line marks this point). That's a solid reduction from the original 30 features while retaining almost all the information.

For reference, just the first 2 components alone capture 63.2% of total variance (PC1: 44.3%, PC2: 19.0%) — which is why even a 2D PCA plot manages to visually separate malignant from benign tumors reasonably well.

Which features contribute most?
PC1 (the dominant axis, 44% of variance) is driven almost entirely by features describing tumor size and shape irregularity:

mean concave points, mean concavity, worst concave points, mean compactness (shape irregularity)
worst perimeter, worst radius, mean perimeter, worst area, mean area (size)
This makes intuitive sense biologically — malignant tumors tend to be larger and have more irregular, concave boundaries, and PC1 essentially captures a "how large and irregular is this tumor" signal, which is also why it correlates so strongly with the malignant/benign label even though PCA never saw the labels.

PC2 is a different mix, most influenced by:

mean fractal dimension, fractal dimension error, worst fractal dimension (texture complexity)
With mean radius, mean area, worst radius showing up with negative loadings — meaning PC2 partly captures a contrast between fractal/texture complexity and raw size.
Practical takeaway: the "worst" and "mean" versions of concavity, concave points, perimeter, radius, and area are the real workhorses of this dataset's variance — the "smoothness," "symmetry," and "texture" features contribute comparatively little to the top components.

In [12]:
# Question 9: A fintech company has:
# 500 features
#  2 million rows
#  severe class imbalance
#Questions:
#1. Would you use PCA? Why?
#2. Would KNN be a good choice?
#3. Which boosting model would you prefer?
#4. How would you reduce computation cost?


- Cautiously yes, but not blindly. With 500 features, dimensionality reduction is worth considering, but PCA has real trade-offs in a fintech context:

Arguments for PCA:

500 features likely have multicollinearity (correlated financial ratios, transaction aggregates, etc.), so a lot of that dimensionality is redundant. PCA can compress this into a smaller set of components that still capture most of the variance.
Reduces training time and memory footprint significantly at 2M rows — this matters a lot at scale.
Can help denoise the data and reduce overfitting risk for distance-based or linear models.

Arguments against / caveats:

Interpretability loss: In fintech (credit risk, fraud detection), regulators and stakeholders often require explainable decisions (e.g., "why was this loan denied" — see regulations like ECOA/Reg B in the US, or GDPR's "right to explanation"). PCA components are linear combinations of original features and are hard to explain to a compliance team or a customer.
Tree-based/boosting models don't need it: If you're going to use XGBoost/LightGBM (very common in fintech), these models handle high dimensionality and irrelevant features natively via feature splitting — PCA usually doesn't help them and can even hurt performance by destroying the original feature semantics that trees exploit well (e.g., a threshold split on "days since last late payment" is meaningful; a threshold on PC7 is not).
PCA can distort minority class signal: Since PCA maximizes overall variance (majority class dominated), it may de-emphasize subtle directions where the rare/minority class actually differs — actively hurting an already-hard imbalanced problem.

My recommendation: Use PCA selectively — e.g., as a preprocessing step only for distance-based or linear models (KNN, logistic regression, SVM), or for visualization/EDA — but keep the raw (or minimally engineered) features for the tree-based/boosting models that will likely be your primary production model.

2. Would KNN be a good choice?

No, I'd avoid KNN here. Several compounding problems:

Curse of dimensionality: With 500 features, distance metrics become close to meaningless — as dimensionality grows, all points tend to become roughly equidistant from each other, which destroys KNN's core assumption that "nearby points are similar."
Scalability: KNN is a lazy learner — at prediction time it has to search through (a large portion of) 2 million rows to find neighbors. Even with KD-trees/ball trees/approximate nearest neighbor methods, this becomes slow and memory-heavy at this scale, especially for real-time fintech applications like fraud scoring, which often need millisecond-level latency.
Class imbalance: KNN is majority-class-biased by nature — for a given query point, the "K nearest neighbors" vote will typically be dominated by the majority class unless you very carefully use distance-weighted voting, resampling, or adjust K. This makes it a poor default for imbalanced problems like fraud/default detection, where the minority class is what you actually care about detecting.

Better alternatives: Tree-based ensembles (Random Forest, Gradient Boosting/XGBoost/LightGBM), regularized logistic regression, or neural nets — all of which handle high dimensionality and large datasets far more gracefully, and have well-established techniques for class imbalance.

3. Which boosting model would you prefer?

For 500 features, 2M rows, and severe imbalance, I'd lean toward LightGBM, with XGBoost as a strong alternative — and here's the reasoning:

Model	Fit for this scenario
LightGBM	Uses histogram-based splitting and leaf-wise tree growth, making it significantly faster and more memory-efficient than traditional GBM on large datasets (2M rows, 500 features is well within its comfort zone). Handles categorical features natively. Has built-in support for imbalance via is_unbalance or scale_pos_weight.
XGBoost	Extremely mature, robust, with excellent regularization (L1/L2) to control overfitting on high-dimensional data, and strong built-in handling of imbalance via scale_pos_weight. Slightly slower than LightGBM at this scale but very well-supported in production/regulated environments (a plus in fintech).
CatBoost	Great if you have a lot of categorical features (common in fintech: merchant category, transaction type, region) since it handles them natively without manual encoding, and has strong out-of-the-box handling of imbalance. Can be slower to train at very large scale compared to LightGBM.
AdaBoost	Generally not preferred here — sensitive to noisy data/outliers (common in financial transaction data), doesn't scale as well to large datasets, and lacks the built-in imbalance-handling and regularization tools of modern gradient boosting libraries.

My pick: LightGBM — best speed/memory profile at 2M rows × 500 features, with is_unbalance=True or a tuned scale_pos_weight, combined with proper evaluation metrics for imbalance (PR-AUC, F1, recall at fixed precision — not raw accuracy).

I'd also pair whichever boosting model is chosen with:

scale_pos_weight (or equivalent) tuned to the imbalance ratio
Possibly SMOTE/undersampling on the training set only (not validation/test) if class weighting alone isn't sufficient
Threshold tuning post-training, since the default 0.5 probability cutoff is rarely optimal under imbalance
4. How would you reduce computation cost?

Several levers, often combined:

Data-level:

Sampling: Use stratified subsampling for initial experimentation/hyperparameter tuning (train on a representative sample, e.g. 200–500K rows) before scaling up to the full 2M for final training.
Undersample the majority class (carefully, with cross-validation) to reduce total training rows while preserving the signal from the rare class — common in fraud/default detection pipelines.
Feature selection before modeling: drop near-zero-variance features, highly correlated features (correlation threshold pruning), and use feature importance from a quick baseline model to cut down from 500 features to a more essential subset (e.g., top 100–150).

Algorithmic:

Histogram-based boosting (LightGBM's default, or XGBoost's tree_method='hist' or gpu_hist) — bins continuous features into discrete buckets, massively speeding up split-finding versus exact greedy methods.
GPU acceleration: Both XGBoost and LightGBM support GPU training, which can cut training time by an order of magnitude at this scale.
Early stopping: Monitor validation performance and stop boosting rounds once improvement plateaus, rather than running a fixed large number of trees.
Reduced hyperparameter search space: Use randomized search or Bayesian optimization (e.g., Optuna) instead of exhaustive grid search, since grid search over many hyperparameters at this data scale is prohibitively expensive.

Infrastructure/engineering:

Distributed training: LightGBM and XGBoost both support distributed training (Spark, Dask) to parallelize across a cluster if a single machine becomes a bottleneck.
Efficient data formats: Store data in columnar, compressed formats (Parquet) rather than CSV, and load with efficient libraries (e.g., pandas with pyarrow backend, or cuDF if using GPU) to reduce I/O overhead.
Incremental/online learning if the data arrives continuously (common in fintech transaction streams) — retrain incrementally rather than from scratch each time.